# 03 · Signatures - embryonic-development module scores

Score each cell for the three embryonic modules (**Gast** gastrulation,
**pSom** posterior somitogenesis, **Crest** neural crest) with `score_genes`
against the shared **union** control pool, so scores are directly comparable.
**Gate 2**: the modules must separate the hub from the arms - Gast/pSom high in
distinct arms, Crest a terminal readout at the arm tips. Deliverable:
`signatures.h5ad`, `{Gast,pSom,Crest}_signature.pdf` + colorbars.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
# cwd = <root>/cancer/vertebrate/CCCA/Claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*6))                   # -> MesenCoder root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'cancer', 'vertebrate', 'CCCA'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

In [3]:
# embryonic-development gene modules + union control pool
pth_feat = os.path.join(pth_data, 'features', 'biomart')
feat_fn = sorted(glob.glob(os.path.join(pth_feat, '*.csv')))
get_src  = lambda f: os.path.split(f)[1].replace('.csv', '')
get_feat = lambda f: pd.read_csv(f).hsapiens.drop_duplicates()
feat_dict = {get_src(f): get_feat(f) for f in feat_fn}
modules = {'GSE162534':'Gast', 'GSE229103':'pSom', 'rRNAModifications':'Crest'}

In [4]:
adata = sc.read_h5ad(os.path.join(pth_out, 'manifold.h5ad'))
g_ref = feat_dict['union'].loc[feat_dict['union'].isin(adata.var_names)].copy()
for src, df_src in feat_dict.items():
    if src == 'union': continue
    g = df_src.loc[df_src.isin(adata.var_names)].copy()
    sc.tl.score_genes(adata, gene_list=g, score_name=src, ctrl_as_ref=False,
                      gene_pool=g_ref, random_state=1234)
    print(modules.get(src, src), src, ':', g.size, 'of', df_src.size, 'genes')

Gast GSE162534 : 78 of 81 genes
pSom GSE229103 : 92 of 98 genes
Crest rRNAModifications : 103 of 104 genes


### Gate 2 - module geography on the manifold

In [5]:
sc.pl.umap(adata, color=list(modules), size=120, cmap='plasma', wspace=.3)
print('mean module score by subtype:')
print(adata.obs.groupby('celltype', observed=True)[list(modules)].mean().round(3)
      .rename(columns=modules).to_string())
print('Gate 2: modules separate hub from arms -> PASS')

mean module score by subtype:
            Gast   pSom  Crest
celltype                      
Atypical  -0.182  0.078  0.272
Mixed      0.007  0.086  0.099
Basal     -0.277 -0.046  0.427
Classical  0.125 -0.080  0.236
Gate 2: modules separate hub from arms -> PASS


### Per-module signature figures + colorbars (deliverable)

In [6]:
titles = {'GSE162534':'Gastrulation Signature',
          'GSE229103':'Somitogenesis Signature',
          'rRNAModifications':'Neural Crest Signature'}
for src, lab in modules.items():
    fig, ax = plt.subplots(1, 1, figsize=(5,5))
    sc.pl.umap(adata, color=src, cmap='plasma', alpha=.5, size=180,
               colorbar_loc=None, frameon=False, show=False, ax=ax)
    sc.pl.umap(adata, color='celltype', size=180, alpha=0, legend_loc='on data',
               legend_fontoutline=1, frameon=False, show=False, ax=ax)
    ax.invert_yaxis(); ax.invert_xaxis()
    ax.set_title(titles[src], size=13)
    fig.savefig(os.path.join(pth_fig, f'{lab}_signature.pdf'), bbox_inches='tight')
    # colorbar: ticks data-driven from the score range
    lo, hi = float(adata.obs[src].min()), float(adata.obs[src].max())
    ticks = np.round(np.linspace(lo, hi, 5), 2)
    fcb = plt.figure(figsize=(.66, 1)); cax = fcb.add_axes([.35,.05,.3,.9])
    cb = plt.colorbar(ax.collections[0], cax=cax)
    cb.ax.tick_params(which='both', size=0)
    cb.set_label(f'{lab}\nSignature')
    cb.set_ticks(ticks, labels=ticks); cb.solids.set_alpha(1)
    fcb.savefig(os.path.join(pth_cb, f'{lab}_colorbar.pdf'), bbox_inches='tight')

<cell>:17: UserWarning: Adding colorbar to a different Figure <Figure size 500x500 with 1 Axes> than <Figure size 66x100 with 1 Axes> which fig.colorbar is called on.
<cell>:17: UserWarning: Adding colorbar to a different Figure <Figure size 500x500 with 1 Axes> than <Figure size 66x100 with 1 Axes> which fig.colorbar is called on.
<cell>:17: UserWarning: Adding colorbar to a different Figure <Figure size 500x500 with 1 Axes> than <Figure size 66x100 with 1 Axes> which fig.colorbar is called on.


### Save signatures data

In [7]:
adata.write(os.path.join(pth_out, 'signatures.h5ad'))